# Reproducibility check: official MedIAnomaly code, AE-d* on RSNA

Runs the **unmodified** official `train.py` / `test.py` of `caiyu6666/MedIAnomaly` for AE-d* on RSNA (latent size 4), to find out whether the gap between our runs (AUROC 70.0 ± 1.1, all training images) and the published result (72.9 ± 2.1) comes from our training loop or from run-to-run variation.

Two sets of three runs:
- **Random seeds**, exactly as the authors' script `train_eval_latent_size.sh` (their `--train-seed` defaults to None, i.e. a random seed per run). This reproduces how the published numbers were obtained.
- **Seeds 0, 1, 2** (`--train-seed`), the seeds of our own runs, for a like-for-like comparison with our loop.

Only two environment changes: wandb logging is disabled (the code calls `wandb.init`), and the RSNA archive is taken from the Drive cache of the earlier notebooks. Each run trains 250 epochs; the AUROC/AP of the final model are read from `test.py`'s `metrics.txt`.

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────
!pip install -q wandb thop SimpleITK medpy scikit-image
import os, re, subprocess, time, glob
import numpy as np, pandas as pd
os.environ['WANDB_MODE'] = 'disabled'          # the official code calls wandb.init(); no account needed
if not os.path.exists('/content/MedIAnomaly'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/caiyu6666/MedIAnomaly.git', '/content/MedIAnomaly'], check=True)
REC = '/content/MedIAnomaly/reconstruction'
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

In [ ]:
# ── Drive and data (RSNA from the cached Zenodo archive) ───────────────
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/STRAP_MedIAnomaly'
TAR = os.path.join(DRIVE_ROOT, 'data_tar', 'RSNA.tar.gz')
DATA_ROOT = os.path.expanduser('~/MedIAnomaly-Data')       # path used by the official code
os.makedirs(DATA_ROOT, exist_ok=True)
if not os.path.isdir(os.path.join(DATA_ROOT, 'RSNA')):
    if not os.path.exists(TAR):
        os.makedirs(os.path.dirname(TAR), exist_ok=True)
        subprocess.run(['wget', '-q', '-O', TAR, 'https://zenodo.org/records/12677223/files/RSNA.tar.gz?download=1'], check=True)
    subprocess.run(['tar', '-xzf', TAR, '-C', DATA_ROOT], check=True)
print(os.listdir(DATA_ROOT))

In [ ]:
# ── Run the official scripts ───────────────────────────────────────────
OUT_CSV = os.path.join(DRIVE_ROOT, 'official_rsna_check.csv')
RUNS = [('random', f'rand{i}', None) for i in range(3)] + [('seeded', f'seed{s}', s) for s in range(3)]
done = pd.read_csv(OUT_CSV) if os.path.exists(OUT_CSV) else pd.DataFrame(columns=['setting', 'fold', 'seed', 'auroc', 'ap'])
for setting, fold, seed in RUNS:
    if fold in set(done['fold']): print('skip', fold); continue
    t0 = time.time()
    args = ['-ls', '4', '-d', 'rsna', '-m', 'ae', '-g', '0', '-f', fold]
    tr = subprocess.run(['python', 'train.py', *args] + (['--train-seed', str(seed)] if seed is not None else []),
                        cwd=REC, capture_output=True, text=True)
    if tr.returncode != 0: print(tr.stderr[-2000:]); raise RuntimeError('train.py failed')
    used_seed = re.search(r'seed:(\d+)', tr.stdout); used_seed = int(used_seed.group(1)) if used_seed else seed
    te = subprocess.run(['python', 'test.py', *args], cwd=REC, capture_output=True, text=True)
    if te.returncode != 0: print(te.stderr[-2000:]); raise RuntimeError('test.py failed')
    auc = float(re.search(r'^AUC: ([0-9.]+)', te.stdout, re.M).group(1)); ap = float(re.search(r'^AP: ([0-9.]+)', te.stdout, re.M).group(1))
    done = pd.concat([done, pd.DataFrame([{'setting': setting, 'fold': fold, 'seed': used_seed, 'auroc': auc, 'ap': ap}])], ignore_index=True)
    done.to_csv(OUT_CSV, index=False)
    print(f'{fold} (seed {used_seed}): AUROC {auc:.4f}  AP {ap:.4f}  ({time.time() - t0:.0f}s)')

In [ ]:
# ── Summary ────────────────────────────────────────────────────────────
r = pd.read_csv(OUT_CSV)
print(r.to_string(index=False)); print()
for s, g in r.groupby('setting'):
    print(f'official code, {s:7s} seeds: AUROC {100*g.auroc.mean():.1f} +/- {100*g.auroc.std():.1f}   AP {100*g.ap.mean():.1f} +/- {100*g.ap.std():.1f}   (n={len(g)})')
print('published (Cai et al., 2024):     AUROC 72.9 +/- 2.1   AP 70.3 +/- 0.9')
print('our loop, all training images:    AUROC 70.0 +/- 1.1   (seeds 0-2)')